# **Extract Bag of Words (BoW) Features from Course Textual Content**


When building a personalized online courses recommender system, it's essential to represent courses in a format that machine learning algorithms can process. Since many items, including online courses, are described by textual data (such as titles and descriptions), we need to transform this raw text into numeric feature vectors.

This analysis focuses on extracting Bag of Words (BoW) features from course titles and descriptions. The BoW approach is a simple but effective technique for characterizing textual data and is widely used in many natural language processing tasks.


## Objectives


This analysis will:

* Extract Bag of Words (BoW) features from course titles and descriptions
* Build a course BoW dataset for content-based recommender system development


----


## Environment Setup


In [2]:
import gensim
import pandas as pd
import nltk as nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from gensim import corpora

%matplotlib inline

Downloading NLTK resources:

In [21]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt to /Users/anthony/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/anthony/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/anthony/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/anthony/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [4]:
# Setting random state for reproducibility
rs = 123

## Understanding Bag of Words (BoW) Features


BoW features represent text as the counts or frequencies of each word that appears in a document. This section demonstrates the process with simple examples.


Consider two example course descriptions:


In [6]:
course1 = "this is an introduction data science course which introduces data science to beginners"
course2 = "machine learning for beginners"
courses = [course1, course2]

### Step 1: Tokenization

The first step in BoW creation is to split the text strings into words (tokens). Tokenization transforms a string into a collection of tokens, which are the smallest units of text such as words or punctuation marks.

In [8]:
# Tokenize the two courses
tokenized_courses = [word_tokenize(course) for course in courses]

### Step 2: Creating a Token Dictionary

Next, we create a token dictionary to index all unique tokens by assigning each one a numeric key/index.

In [13]:
# Create a token dictionary for the two courses
tokens_dict = gensim.corpora.Dictionary(tokenized_courses)
print(tokens_dict.token2id)

{'an': 0, 'beginners': 1, 'course': 2, 'data': 3, 'introduces': 4, 'introduction': 5, 'is': 6, 'science': 7, 'this': 8, 'to': 9, 'which': 10, 'for': 11, 'learning': 12, 'machine': 13}


### Step 3: Generating BoW Features

With the token dictionary in place, we can generate BoW feature vectors by counting token occurrences in each document.

In [14]:
# Generate BoW features for each course
courses_bow = [tokens_dict.doc2bow(course) for course in tokenized_courses]
courses_bow

[[(0, 1),
  (1, 1),
  (2, 1),
  (3, 2),
  (4, 1),
  (5, 1),
  (6, 1),
  (7, 2),
  (8, 1),
  (9, 1),
  (10, 1)],
 [(1, 1), (11, 1), (12, 1), (13, 1)]]

The output consists of two arrays of tuples. Each tuple contains a token ID and its count in the document. For example, `(0, 1)` means token ID 0 (`an`) appears once, and `(7, 2)` means token ID 7 (`science`) appears twice.

Printing each token and its count for better readability:

In [15]:
# Enumerate through each course and its bag-of-words representation
for course_idx, course_bow in enumerate(courses_bow):
    # Print the index of the current course and a label
    print(f"Bag of words for course {course_idx}:")
    # For each token index, print its bow value (word count)
    for token_index, token_bow in course_bow:
        # Retrieve the token from the tokens dictionary based on its index
        token = tokens_dict.get(token_index)
        # Print the token and its bag-of-words value
        print(f"--Token: '{token}', Count:{token_bow}")

Bag of words for course 0:
--Token: 'an', Count:1
--Token: 'beginners', Count:1
--Token: 'course', Count:1
--Token: 'data', Count:2
--Token: 'introduces', Count:1
--Token: 'introduction', Count:1
--Token: 'is', Count:1
--Token: 'science', Count:2
--Token: 'this', Count:1
--Token: 'to', Count:1
--Token: 'which', Count:1
Bag of words for course 1:
--Token: 'beginners', Count:1
--Token: 'for', Count:1
--Token: 'learning', Count:1
--Token: 'machine', Count:1


## BoW Dimensionality Reduction


A document may contain thousands of words, resulting in high-dimensional BoW vectors. To reduce dimensionality, we can filter out common but relatively meaningless tokens such as stop words.

In [17]:
stop_words = set(stopwords.words('english'))

Filtering English stop words from the tokens in course1:

In [18]:
# Tokens in course 1
tokenized_courses[0]

['this',
 'is',
 'an',
 'introduction',
 'data',
 'science',
 'course',
 'which',
 'introduces',
 'data',
 'science',
 'to',
 'beginners']

In [19]:
processed_tokens = [w for w in tokenized_courses[0] if not w.lower() in stop_words]
processed_tokens

['introduction',
 'data',
 'science',
 'course',
 'introduces',
 'data',
 'science',
 'beginners']

Another approach is to keep only nouns in the text, using NLTK's part-of-speech tagging:

In [22]:
tags = nltk.pos_tag(tokenized_courses[0])
tags

[('this', 'DT'),
 ('is', 'VBZ'),
 ('an', 'DT'),
 ('introduction', 'NN'),
 ('data', 'NNS'),
 ('science', 'NN'),
 ('course', 'NN'),
 ('which', 'WDT'),
 ('introduces', 'VBZ'),
 ('data', 'NNS'),
 ('science', 'NN'),
 ('to', 'TO'),
 ('beginners', 'NNS')]

The output shows that words like 'introduction', 'data', 'science', 'course', and 'beginners' are nouns, which could be kept as meaningful tokens in the BoW feature vector.

## Extracting BoW Features from Course Content Dataset


Now we'll apply these techniques to extract BoW features from actual course content data:

In [26]:
course_content_df = pd.read_csv("data/course_processed.csv")
course_content_df.head()

,COURSE_ID,TITLE,DESCRIPTION
0,ML0201EN,robots are coming build iot apps with watson ...,have fun with iot and learn along the way if ...
1,ML0122EN,accelerating deep learning with gpu,training complex deep learning models with lar...
2,GPXX0ZG0EN,consuming restful services using the reactive ...,learn how to use a reactive jax rs client to a...
3,RP0105EN,analyzing big data in r using apache spark,apache spark is a popular cluster computing fr...
4,GPXX0Z2PEN,containerizing packaging and running a sprin...,learn how to containerize package and run a ...


The course content dataset contains three columns: `COURSE_ID`, `TITLE`, and `DESCRIPTION`. We'll extract BoW features from the text in both `TITLE` and `DESCRIPTION`.

### Joining Title and Description Text

In [29]:
# Merge TITLE and DESCRIPTION title
course_content_df['course_texts'] = course_content_df[['TITLE', 'DESCRIPTION']].agg(' '.join, axis=1)
course_content_df = course_content_df.reset_index()
course_content_df['index'] = course_content_df.index
course_content_df.head()

,level_0,index,COURSE_ID,TITLE,DESCRIPTION,course_texts
0,0,0,ML0201EN,robots are coming build iot apps with watson ...,have fun with iot and learn along the way if ...,robots are coming build iot apps with watson ...
1,1,1,ML0122EN,accelerating deep learning with gpu,training complex deep learning models with lar...,accelerating deep learning with gpu training c...
2,2,2,GPXX0ZG0EN,consuming restful services using the reactive ...,learn how to use a reactive jax rs client to a...,consuming restful services using the reactive ...
3,3,3,RP0105EN,analyzing big data in r using apache spark,apache spark is a popular cluster computing fr...,analyzing big data in r using apache spark apa...
4,4,4,GPXX0Z2PEN,containerizing packaging and running a sprin...,learn how to containerize package and run a ...,containerizing packaging and running a sprin...


### Defining a Tokenization Function

In [30]:
def tokenize_course(course, keep_only_nouns=True):
    # Get English stop words
    stop_words = set(stopwords.words('english'))
    # Tokenize the course text
    word_tokens = word_tokenize(course)
    # Remove English stop words and numbers
    word_tokens = [w for w in word_tokens if (not w.lower() in stop_words) and (not w.isnumeric())]
    # Only keep nouns 
    if keep_only_nouns:
        # Define a filter list of non-noun POS tags
        filter_list = ['WDT', 'WP', 'WRB', 'FW', 'IN', 'JJR', 'JJS', 'MD', 'PDT', 'POS', 'PRP', 'RB', 'RBR', 'RBS',
                       'RP']
        # Tag the word tokens with POS tags
        tags = nltk.pos_tag(word_tokens)
        # Filter out non-nouns based on POS tags
        word_tokens = [word for word, pos in tags if pos not in filter_list]

    return word_tokens

Testing the tokenization function on the first course:

In [31]:
a_course = course_content_df.iloc[0, :]['course_texts']
a_course

'robots are coming  build iot apps with watson  swift  and node red have fun with iot and learn along the way  if you re a swift developer and want to learn more about iot and watson ai services in the cloud  raspberry pi   and node red  you ve found the right place  you ll build iot apps to read temperature data  take pictures with a raspcam  use ai to recognize the objects in those pictures  and program an irobot create 2 robot  '

In [32]:
tokenize_course(a_course)

['robots',
 'coming',
 'build',
 'iot',
 'apps',
 'watson',
 'swift',
 'red',
 'fun',
 'iot',
 'learn',
 'way',
 'swift',
 'developer',
 'want',
 'learn',
 'iot',
 'watson',
 'ai',
 'services',
 'cloud',
 'raspberry',
 'pi',
 'node',
 'red',
 'found',
 'place',
 'build',
 'iot',
 'apps',
 'read',
 'temperature',
 'data',
 'take',
 'pictures',
 'raspcam',
 'use',
 'ai',
 'recognize',
 'objects',
 'pictures',
 'program',
 'irobot',
 'create',
 'robot']

### Tokenizing All Courses

In [33]:
data_copy = course_content_df.copy()
for i in range(0, course_content_df.shape[0]):
    course_content_df.loc[i, "course_texts"] = ' '.join(tokenize_course(course_content_df.loc[i, "course_texts"], True))
course_content_df

,level_0,index,COURSE_ID,TITLE,DESCRIPTION,course_texts
0,0,0,ML0201EN,robots are coming build iot apps with watson ...,have fun with iot and learn along the way if ...,robots coming build iot apps watson swift red ...
1,1,1,ML0122EN,accelerating deep learning with gpu,training complex deep learning models with lar...,accelerating deep learning gpu training comple...
2,2,2,GPXX0ZG0EN,consuming restful services using the reactive ...,learn how to use a reactive jax rs client to a...,consuming restful services using reactive jax ...
3,3,3,RP0105EN,analyzing big data in r using apache spark,apache spark is a popular cluster computing fr...,analyzing big data r using apache spark apache...
4,4,4,GPXX0Z2PEN,containerizing packaging and running a sprin...,learn how to containerize package and run a ...,containerizing packaging running spring boot a...
...,...,...,...,...,...,...
302,302,302,excourse89,javascript jquery and json,in this course we ll look at the javascript l...,javascript jquery json course look javascript ...
303,303,303,excourse90,programming foundations with javascript html ...,learn foundational programming concepts e g ...,programming foundations javascript html css le...
304,304,304,excourse91,front end web development with react,this course explores javascript based front en...,front end web development react course explore...
305,305,305,excourse92,introduction to web development,this course is designed to start you on a path...,introduction web development course designed s...


### Creating a Token Dictionary

In [34]:
course_text_tokens = [word_tokenize(seq) for seq in course_content_df["course_texts"].to_list()]
course_dict = gensim.corpora.Dictionary(course_text_tokens)

### Generating BoW Features

In [35]:
bow_list = [course_dict.doc2bow(lst) for lst in course_text_tokens]

### Creating a BoW DataFrame

In [36]:
course_content_df

,level_0,index,COURSE_ID,TITLE,DESCRIPTION,course_texts
0,0,0,ML0201EN,robots are coming build iot apps with watson ...,have fun with iot and learn along the way if ...,robots coming build iot apps watson swift red ...
1,1,1,ML0122EN,accelerating deep learning with gpu,training complex deep learning models with lar...,accelerating deep learning gpu training comple...
2,2,2,GPXX0ZG0EN,consuming restful services using the reactive ...,learn how to use a reactive jax rs client to a...,consuming restful services using reactive jax ...
3,3,3,RP0105EN,analyzing big data in r using apache spark,apache spark is a popular cluster computing fr...,analyzing big data r using apache spark apache...
4,4,4,GPXX0Z2PEN,containerizing packaging and running a sprin...,learn how to containerize package and run a ...,containerizing packaging running spring boot a...
...,...,...,...,...,...,...
302,302,302,excourse89,javascript jquery and json,in this course we ll look at the javascript l...,javascript jquery json course look javascript ...
303,303,303,excourse90,programming foundations with javascript html ...,learn foundational programming concepts e g ...,programming foundations javascript html css le...
304,304,304,excourse91,front end web development with react,this course explores javascript based front en...,front end web development react course explore...
305,305,305,excourse92,introduction to web development,this course is designed to start you on a path...,introduction web development course designed s...


In [37]:
bow_df = pd.DataFrame({"doc_index": [], "doc_id": [], "token": [], "bow": []})
for doc_index, doc_bow in enumerate(bow_list):
    for token_id, occurrences in doc_bow:
        new_row = pd.DataFrame({"doc_index": [doc_index],
                               "doc_id": [course_content_df.loc[doc_index, "COURSE_ID"]],
                               "token": [course_dict.get(token_id)],
                               "bow": occurrences})
        bow_df = pd.concat([bow_df, new_row], ignore_index = True)
bow_df["doc_index"] = bow_df["doc_index"].astype(int)
bow_df["bow"] = bow_df["bow"].astype(int)

In [38]:
bow_df

,doc_index,doc_id,token,bow
0,0,ML0201EN,ai,2
1,0,ML0201EN,apps,2
2,0,ML0201EN,build,2
3,0,ML0201EN,cloud,1
4,0,ML0201EN,coming,1
...,...,...,...,...
10358,306,excourse93,modifying,1
10359,306,excourse93,objectives,1
10360,306,excourse93,pieces,1
10361,306,excourse93,plugins,1


## Advanced Textual Features


Beyond simple Bag of Words features, there are other advanced textual representation techniques worth exploring:

1. **TF-IDF (Term Frequency–Inverse Document Frequency)**: Similar to BoW, TF-IDF counts word frequencies but also adjusts for the frequency of words across all documents. This helps emphasize words that are important to a specific document but not common across all documents.

2. **Text Embedding Vectors**: These techniques use neural networks to learn latent feature representations of textual data. Methods like Word2Vec, GloVe, or sentence transformers can represent words, sentences, or documents as dense vectors that capture semantic relationships.


## Conclusion


This analysis has demonstrated how to extract Bag of Words features from course titles and descriptions. The resulting feature vectors create a numerical representation of textual content that can be used for various machine learning tasks, such as similarity measurements, clustering, or classification.

The BoW feature dataset constructed here provides a foundation for building content-based recommendation systems that can suggest courses based on textual similarity.
